# Early Stopping — Complete Theory

## The Problem It Solves

Training for a fixed number of epochs is wasteful and risky. If you set `num_epochs=100` but the model stops improving at epoch 23, you've wasted 77 epochs of compute. Worse — after epoch 23, the model might start *overfitting* (memorizing training data, getting worse on val).

Early stopping watches your validation metric and stops training automatically when the model stops improving.

---

## The Algorithm

Early stopping has three components:

**1. Patience** — how many epochs of no improvement to tolerate before stopping. With `patience=5`, you wait 5 consecutive bad epochs before giving up.

**2. Best score tracker** — the best val loss seen so far. Every epoch you ask: "did we beat the best?"

**3. Counter** — counts consecutive epochs without improvement. Resets to 0 when improvement is found. When it hits patience, stop.

``` python
best_val_loss = infinity
counter = 0

each epoch:
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0          ← reset, we're still improving
        save checkpoint
    else:
        counter += 1         ← one more bad epoch
        if counter >= patience:
            STOP TRAINING    ← break out of epoch loop
```

---

## The Delta Threshold

Raw numbers have noise. Val loss of `0.8501` vs `0.8499` isn't meaningful improvement — it's floating point variance. A `min_delta` threshold prevents the counter from resetting on trivially small improvements:

```python
if val_loss < best_val_loss - min_delta:
    # genuine improvement
else:
    # not good enough, increment counter
```

Typical values: `min_delta=0.001` or `0.0001`.

---

## Implementation Pattern — As a Class

Early stopping logic is cleanly packaged as a reusable class. The class holds all the state (counter, best score) and exposes one method you call each epoch that returns whether to stop.

**Structure:**
``` python
class EarlyStopping:
    __init__(patience, min_delta)  → store config, initialize counter and best_loss
    __call__(val_loss)             → update state, return True if should stop
```

Using `__call__` means you can use the object like a function:
```python
early_stop = EarlyStopping(patience=5, min_delta=0.001)

if early_stop(avg_val_loss):
    print("Early stopping triggered")
    break
```

---

## Where It Goes in the Loop

``` python
for epoch in range(num_epochs):
    # training loop
    # validation loop

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        save checkpoint

    if early_stop(avg_val_loss):
        break                    ← exits the epoch loop cleanly
```

Note that checkpointing and early stopping are separate concerns — checkpointing saves the best model, early stopping decides when to quit. They work together but independently.

---

## Choosing Patience

Too low → stops before the model has a chance to recover from a temporary plateau.
Too high → defeats the purpose, trains too long.

Rule of thumb: set patience to roughly 10-20% of your total expected epochs. If you'd train for 50 epochs, `patience=5` to `patience=10` is reasonable.

---

## Overfitting looks like this:

``` t
Epoch 20: Train Loss 0.65 | Val Loss 0.71  ← gap starting
Epoch 25: Train Loss 0.58 | Val Loss 0.74  ← val getting WORSE
Epoch 30: Train Loss 0.51 | Val Loss 0.79  ← classic overfit
```

Val loss is *increasing* — which means it's not improving — which means the early stopping counter increments every epoch — which means early stopping triggers automatically when model starts to overfit.

The checkpoint separately ensures you always have the best model saved, not the final overfitted one.

---

## TODO 1: Implement the EarlyStopping Class

**Goal:** Build a reusable `EarlyStopping` class from scratch.

**Requirements:**
- `__init__` takes `patience` and `min_delta=0.0`
- Tracks `best_loss`, `counter`, and `should_stop` as instance variables
- `__call__(val_loss)` updates state and returns `True` if training should stop
- Prints a message each time: either "improved" or "no improvement (counter/patience)"

**Hints:**
- Initialize `best_loss = float('inf')`
- The comparison is `val_loss < best_loss - min_delta`
- Return `self.should_stop` at the end of `__call__`

---

## TODO 2: Integrate Into Training Loop

**Goal:** Add early stopping to your full training loop alongside checkpointing.

**Requirements:**
- Instantiate `EarlyStopping(patience=5, min_delta=0.001)` before the loop
- Call it after validation, after the scheduler step
- If it returns `True`, print which epoch training stopped at and break
- Run for `num_epochs=50` so early stopping actually has a chance to trigger
- Checkpointing should still work independently alongside it

**Expected output when triggered:**
``` t
Early stopping triggered at epoch 23
Best val loss achieved: 0.7812
```

In [23]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import os
import sys
import json

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


### loading the sample

In [24]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path, limit=None):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            full_data = json.load(f)
        
        if limit:
            self.data = full_data[:limit]
        else:
            self.data = full_data
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

### getting the path to the dataset and loading the metadata

In [25]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path, 2000)
print(len(val_dataset))

test_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "test" / "test_manifest.json"
test_dataset = SpeakerDataset(test_manifest_path)
print(len(test_dataset))


10000
2000
10000


In [26]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### creating the neural network

In [27]:
class SpeakerCounter(nn.Module):
    """
    Classifier that predicts number of speakers (1, 2, or 3)
    from audio features.
    """
    def __init__(self, input_size=5, hidden_sizes=[64, 32, 16], num_classes=3):
        super().__init__()
        # TODO: Build network with given architecture
        # Hint: Use nn.Sequential or define layers individually
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_sizes[0],hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_sizes[1],hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_sizes[2],num_classes)
        )
        
    def forward(self, x):
        # TODO: Implement forward pass
        return self.network(x)

### calculating the loss

In [28]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

### todo 1

In [29]:
class EarlyStopping:
    def __init__(self, patience, min_delta):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float("inf")
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.counter = 0
            self.best_loss = val_loss
            self.should_stop = False # improvement 
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            else: 
                self.should_stop = False
        
        return self.should_stop

In [30]:
project_root = current_dir.parent.parent.parent
checkpoint_save1 = project_root / "notebooks" / "phase2notes" / "2_4_training_minior_topics" / "checkpoints" / "2_4_4" / "todo"
checkpoint_save1.mkdir(parents=True, exist_ok=True)
checkpoint_save1 = checkpoint_save1 / "checkpoint.pt"

### todo 2

In [31]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6)
best_val_loss = float('inf')
num_epochs = 70
early_stop = EarlyStopping(patience=5, min_delta=0.001)

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch_train = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch_train} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch_train
    avg_train_acc = running_train_acc / steps_per_epoch_train    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0     
    steps_per_epoch_val = len(val_loader)
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 30 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch_val} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / steps_per_epoch_val
    avg_val_acc = running_val_acc / steps_per_epoch_val   
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_acc': avg_val_acc,
        }
        torch.save(checkpoint, checkpoint_save1)
        print(f"new best! Saved CheckPoint (loss:{avg_val_loss:.6f})")
    else:
        print("No improvement")

    if early_stop(avg_val_loss):
        print("Early stopping triggered")
        print(f"triggered at epoch: {epoch}")
        print(f"best val loss achieved: {early_stop.best_loss:.6f}")
        break




  train Batch 50/312 | Loss: 1.1964
  train Batch 100/312 | Loss: 1.1690
  train Batch 150/312 | Loss: 1.3431
  train Batch 200/312 | Loss: 1.2634
  train Batch 250/312 | Loss: 1.1685
  train Batch 300/312 | Loss: 1.2450
  val Batch 30/63 | Loss: 1.0498
  val Batch 60/63 | Loss: 1.0616

--- Epoch 1/70 Summary ---
Train Loss: 1.2022 | Train Acc: 37.04%
Val Loss:   1.1043 | Val Acc:   41.91%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:1.104252)
  train Batch 50/312 | Loss: 1.1819
  train Batch 100/312 | Loss: 1.1355
  train Batch 150/312 | Loss: 1.1355
  train Batch 200/312 | Loss: 1.0603
  train Batch 250/312 | Loss: 1.0560
  train Batch 300/312 | Loss: 0.9803
  val Batch 30/63 | Loss: 0.9714
  val Batch 60/63 | Loss: 0.9790

--- Epoch 2/70 Summary ---
Train Loss: 1.0606 | Train Acc: 46.75%
Val Loss:   1.0070 | Val Acc:   48.86%
Current Learning Rate: 0.0001

new best! Saved CheckPoint (loss:1.006982)
  train Batch 50/312 | Loss: 1.1537
  train Batch 100/312 | Loss: 